In [1]:
import ROOT as rt
# import root_numpy as rtnp
import csv
import re
import sys
import collections

from collections import OrderedDict
import uproot
import pandas as pd

import scipy
import awkward as ak
import numpy as np
import time
import numba
from numba import jit
from matplotlib import pyplot as plt
sys.path.append('/users/lisa.benato/private/analysis_notebooks/SUEPs_FWF_feasibility/lib/')
from histo_utilities import create_TH1D, create_TH2D, std_color_list, create_TGraph, make_ratio_plot
from scipy.stats import norm

import CMS_lumi, tdrstyle
a = tdrstyle.setTDRStyle()
CMS_lumi.writeExtraText = 0



# donotdelete = []
print(sys.version)

ERROR in cling::CIFactory::createCI(): cannot extract standard library include paths!
Invoking:
  LC_ALL=C x86_64-conda-linux-gnu-c++   -DNDEBUG -xc++ -E -v /dev/null 2>&1 | sed -n -e '/^.include/,${' -e '/^ \/.*++/p' -e '}'
Results was:
With exit code 0


Welcome to JupyROOT 6.24/06
3.7.12 | packaged by conda-forge | (default, Oct 26 2021, 06:08:53) 
[GCC 9.4.0]


# Load ntuples

In [7]:
fpath_bkg =OrderedDict()
tree_bkg = OrderedDict()
tree_sig = OrderedDict()
fpath_sig =OrderedDict()


start_t = time.time()



isData = 0
if isData:
    ntupler_version = 'V1p17/'
    
    analyzer_version = "/v5/v167/"


    data_path = '/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/csc/'+ntupler_version+'/Data2018/'+analyzer_version+'/normalized/'
    
    fpath_bkg['data'] = data_path + 'Run2_displacedJetMuonNtupler_V1p17_Data2016_Data2017_Data2018-HighMET_goodLumi.root'
#     fpath_bkg['data18'] = data_path + 'Run2_displacedJetMuonNtupler_V1p17_Data2018_17Sept2018_Run2018-HighMET-17Sep2018_goodLumi.root'
#     data_path = '/mnt/hadoop/store/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/csc/'+ntupler_version+'/Data2017/'+analyzer_version+'/normalized/'
#     fpath_bkg['data17'] = data_path + 'Run2_displacedJetMuonNtupler_V1p17_Data2017_Run2017-HighMET-17Nov2017_goodLumi.root'
#     data_path = '/mnt/hadoop/store/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/csc/'+ntupler_version+'/Data2016/'+analyzer_version+'/normalized/'
#     fpath_bkg['data16'] = data_path + 'Run2_displacedJetMuonNtupler_V1p17_Data2016_Run2016-HighMET-07Aug17_goodLumi.root'



    

else:
    ntupler_version = 'V1p17/'

    analyzer_version = "/v1/v124/" #includes DT
    #path = ' /storage/cms/store/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/csc//V1p17/'
    path = '/storage/af/user/christiw/login-1/christiw/LLP/displacedJetMuonAnalyzer/csc/'


    decay = 'bbbb'
    mass = '55'
    lumi = [ 35920, 41530, 59740 ]
    years = ['MC_Summer16', 'MC_Fall17', 'MC_Fall18']
    tune = ['TuneCUETP8M1_13TeV-powheg-pythia8','TuneCP5_13TeV-powheg-pythia8','TuneCP5_13TeV-powheg-pythia8']
    ctaus  = ['100','1000','10000','100000']
    year_index = 0
#     for ct in ctaus:
        
#         if year_index == 3:
#             mc_path = path+ntupler_version+'/MC_all/'+analyzer_version+'/normalized/'
#             fpath_bkg['m'+mass+'ctau'+ct] = mc_path + 'ggH_HToSSTo'+decay+'_MH-125_MS-'+str(mass)+'_ctau-'+ct+'_137000pb_weighted.root'
#         else:
#             mc_path = path+ntupler_version+'/'+years[year_index]+'/'+analyzer_version+'/normalized/'
#             fpath_bkg['m'+mass+'ctau'+ct] = mc_path + 'ggH_HToSSTo'+decay+'_MH-125_MS-'+str(mass)+'_ctau-'+ct+'_'+tune[year_index]+'_'+str(lumi[year_index])+'pb_weighted.root'

            
    fpath_bkg['test'] = '/users/lisa.benato/private/DelphesMDS/CMSSW_9_4_4/src/llp_analyzer/MuonSystem_Tree_large.root'
NEvents = {}
Total = {}
accep = {}
accep_met = {}
for k,v in fpath_bkg.items():
    print (k, v)
    root_dir = uproot.open(v) 
    tree_bkg[k] = root_dir['MuonSystem']
    NEvents[k] = root_dir['NEvents'].values()[0]#NEvents[k] = root_dir['NEvents'][1]
    if not isData:
        #Total[k] = root_dir['Total'].values()[0]#[1]
        accep[k] = root_dir['acceptance'].values()[0]#[1]
        accep_met[k] = root_dir['acceptance_met'].values()[0]#[1]

    a = tree_bkg[k]["weight"].array()
    print("NEvents",NEvents[k],len(a))
    print("accep",accep[k],len(a))
    print("accep_met",accep_met[k],len(a))


test /users/lisa.benato/private/DelphesMDS/CMSSW_9_4_4/src/llp_analyzer/MuonSystem_Tree_large.root
NEvents 7307043.0 340000
accep 5804984.5 340000
accep_met 5804984.5 340000


# background cut flow

In [8]:
JET_PT_CUT = 10.0
MUON_PT_CUT = 10.0
N_RECHIT_CUT = 100
jetPt_cut = 30

weight = {}
sel_ev = {}

ncluster = 1
cluster_index = '3'
table = OrderedDict()
table['acc'] = "Acceptance "
table['met'] = "Trigger and MET cut "
table['met_filter'] = "MET filters "



table['lep'] = "$N_{lepton} = 0$ "
table['jet'] = r"$N_{jet} \geq 1$ "
table['dphi_jetmet'] = 'dphi(jet, met)'
table['dt_station'] = 'dt_station<3'
table['dt_wheel'] = 'dt_wheel<3'

table['1_cluster'] = "$N_{cluster} \geq 1$ "
table['jet_veto'] = "jet veto "
table['muon_veto'] = "muon veto "
table['mb1'] = "MB1 veto "
table['rpc'] = "rpc matching "
table['mb1_adjacent'] = "mb1_adjacent "
table['dphi'] = r"$\Delta\phi\mathrm{ (cluster,MET)}$ "
table['nrechits'] = r"$N_{rechits}$ cut "

import awkward as ak
import numpy as np

for k, T in tree_bkg.items():

    # load as awkward arrays (CRUCIAL)
    arr = T.arrays(library="ak")

    ########### SELECTION: JETS ############
    jetPt  = arr["jetPt"]
    jetEta = arr["jetEta"]
    jetID  = arr["jetTightPassId"]

    sel_jet = (jetPt >= jetPt_cut) & (abs(jetEta) < 2.4) & (abs(jetID))

    ########### SELECTION: EVENTS ############
    total = NEvents[k]

    sel_ev[k] = arr["METNoMuTrigger"]
    sel_ev[k] = sel_ev[k] & (arr["metEENoise"] >= 200)

    accep_met[k] = ak.sum(sel_ev[k])
    print("trigger + MET",
          accep_met[k],
          accep_met[k]/total,
          accep_met[k]/total,
          ak.sum(sel_ev[k]))

    table["met"] += " & {0:6.3f} & {1:6.3f} & {2:6.0f}".format(
        100*accep_met[k]/total,
        100*accep_met[k]/total,
        ak.sum(sel_ev[k])
    )

    acc_met = ak.sum(sel_ev[k])
    sel_temp = ak.sum(sel_ev[k])

    sel_ev[k] = sel_ev[k] & (arr["Flag2_all"] == 1)

    print("MET filters",
          ak.sum(sel_ev[k])/acc_met,
          ak.sum(sel_ev[k])/sel_temp,
          ak.sum(sel_ev[k]))

    table["met_filter"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k])/sel_temp,
        100*ak.sum(sel_ev[k])/acc_met,
        ak.sum(sel_ev[k])
    )

    sel_temp = ak.sum(sel_ev[k])

    # at least 1 selected jet
    sel_ev[k] = sel_ev[k] & (ak.sum(sel_jet, axis=1) >= 1)

    print("1 jet",
          ak.sum(sel_ev[k])/acc_met,
          ak.sum(sel_ev[k])/sel_temp,
          ak.sum(sel_ev[k]))

    table["jet"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k])/sel_temp,
        100*ak.sum(sel_ev[k])/acc_met,
        ak.sum(sel_ev[k])
    )

    sel_temp = ak.sum(sel_ev[k])

    sel_ev[k] = sel_ev[k] & (abs(arr["jetMet_dPhiMin"]) > 0.6)

    print("dphi(jet, met)",
          ak.sum(sel_ev[k])/acc_met,
          ak.sum(sel_ev[k])/sel_temp,
          ak.sum(sel_ev[k]))

    table["dphi_jetmet"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k])/sel_temp,
        100*ak.sum(sel_ev[k])/acc_met,
        ak.sum(sel_ev[k])
    )

    sel_temp = ak.sum(sel_ev[k])

    sel_ev[k] = sel_ev[k] & (arr["nDtStations25"] < 3)

    print("DT station < 3",
          ak.sum(sel_ev[k])/acc_met,
          ak.sum(sel_ev[k])/sel_temp,
          ak.sum(sel_ev[k]))

    table["dt_station"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k])/sel_temp,
        100*ak.sum(sel_ev[k])/acc_met,
        ak.sum(sel_ev[k])
    )

    sel_temp = ak.sum(sel_ev[k])

    sel_ev[k] = sel_ev[k] & (arr["nDtWheels25"] < 3)

    print("DT wheel < 3",
          ak.sum(sel_ev[k])/acc_met,
          ak.sum(sel_ev[k])/sel_temp,
          ak.sum(sel_ev[k]))

    table["dt_wheel"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k])/sel_temp,
        100*ak.sum(sel_ev[k])/acc_met,
        ak.sum(sel_ev[k])
    )

    sel_ev[k] = sel_ev[k] & ((arr["runNum"] < 275750) | (arr["runNum"] > 275950))

    table["lep"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k])/sel_temp,
        100*ak.sum(sel_ev[k])/acc_met,
        ak.sum(sel_ev[k])
    )

    sel_temp = ak.sum(sel_ev[k])

    ########### SELECTION: CLUSTERS ############

    clusterSize = arr["dtRechitClusterSize"]
    clusterPhi  = arr["dtRechitClusterMetEENoise_dPhi"]

    binD = (clusterSize >= 100) & (abs(clusterPhi) < 1)

    sel_rechitcluster = abs(arr["dtRechitClusterX"]) >= 0
    sel_rechitcluster = sel_rechitcluster & (~binD)

    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    print("cluster efficiency",
          ak.sum(sel_ev[k] & has_cluster)/acc_met,
          ak.sum(sel_ev[k] & has_cluster)/sel_temp)

    table["1_cluster"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k] & has_cluster)/sel_temp,
        100*ak.sum(sel_ev[k] & has_cluster)/acc_met,
        ak.sum(sel_ev[k] & has_cluster)
    )

    sel_temp = ak.sum(sel_ev[k] & has_cluster)

    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitClusterJetVetoPt"] < JET_PT_CUT)
    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    print("jet cut",
          ak.sum(sel_ev[k] & has_cluster)/acc_met,
          ak.sum(sel_ev[k] & has_cluster)/sel_temp)

    table["jet_veto"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k] & has_cluster)/sel_temp,
        100*ak.sum(sel_ev[k] & has_cluster)/acc_met,
        ak.sum(sel_ev[k] & has_cluster)
    )

    sel_temp = ak.sum(sel_ev[k] & has_cluster)

    sel_rechitcluster = sel_rechitcluster & ~(
        (arr["dtRechitClusterMuonVetoPt"] >= 10) &
        (arr["dtRechitClusterMuonVetoLooseId"])
    )

    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    print("muon cut",
          ak.sum(sel_ev[k] & has_cluster)/acc_met,
          ak.sum(sel_ev[k] & has_cluster)/sel_temp)

    table["muon_veto"] += " & {0:6.2f} & {1:6.2f} & {2:6.0f}".format(
        100*ak.sum(sel_ev[k] & has_cluster)/sel_temp,
        100*ak.sum(sel_ev[k] & has_cluster)/acc_met,
        ak.sum(sel_ev[k] & has_cluster)
    )
    
for k,v in table.items():
    print(v+r" \\")

trigger + MET 2375 0.00032502887967129793 0.00032502887967129793 2375
MET filters 0.9898947368421053 0.9898947368421053 2351
1 jet 0.96 0.9698000850701829 2280
dphi(jet, met) 0.8522105263157894 0.887719298245614 2024
DT station < 3 0.8349473684210527 0.9797430830039525 1983
DT wheel < 3 0.832421052631579 0.9969742813918305 1977
cluster efficiency 0.058105263157894736 0.06980273141122914
jet cut 0.03494736842105263 0.6014492753623188
muon cut 0.03494736842105263 1.0
Acceptance  \\
Trigger and MET cut  &  0.033 &  0.033 &   2375 \\
MET filters  &  98.99 &  98.99 &   2351 \\
$N_{lepton} = 0$  &  99.70 &  83.24 &   1977 \\
$N_{jet} \geq 1$  &  96.98 &  96.00 &   2280 \\
dphi(jet, met) &  88.77 &  85.22 &   2024 \\
dt_station<3 &  97.97 &  83.49 &   1983 \\
dt_wheel<3 &  99.70 &  83.24 &   1977 \\
$N_{cluster} \geq 1$  &   6.98 &   5.81 &    138 \\
jet veto  &  60.14 &   3.49 &     83 \\
muon veto  & 100.00 &   3.49 &     83 \\
MB1 veto  \\
rpc matching  \\
mb1_adjacent  \\
$\Delta\phi\math

# signal cut flow

In [ ]:
import awkward as ak
import numpy as np

JET_PT_CUT = 20.0
MUON_PT_CUT = 10.0
N_RECHIT_CUT = 100
jetPt_cut = 30

weight = {}
sel_ev = {}

ncluster = 1
cluster_index = '3'
table = OrderedDict()
table['top'] = "Selection - sel/prev - sel/met - surviving entry"
table['acc'] = "Acceptance "
table['met'] = "Trigger and MET cut "
table['met_filter'] = "MET filters "



table['lep'] = "$N_{lepton} = 0$ "
table['jet'] = r"$N_{jet} \geq 1$ "
table['dphi_jetmet'] = 'dphi(jet, met)'
table['dt_station'] = 'dt_station<3'
table['dt_wheel'] = 'dt_wheel<3'

table['1_cluster'] = "$N_{cluster} \geq 1$ "
table['jet_veto'] = "jet veto "
table['muon_veto'] = "muon veto "
table['mb1'] = "MB1 veto "
table['rpc'] = "rpc matching "
table['mb1_adjacent'] = "mb1_adjacent "
table['dphi'] = r"$\Delta\phi\mathrm{ (cluster,MET)}$ "
table['nrechits'] = r"$N_{rechits}$ cut "
table['mb2'] = 'mb2'

for k, T in tree_bkg.items():
    if '100000' in k:
        continue

    # load arrays
    arr = T.arrays(library="ak")

    ########### WEIGHTS ############
    weight = arr["weight"]
    weight = ak.ones_like(arr["weight"])
    # optionally:
    # weight = arr["weight"] * arr["pileupWeight"]

    print(weight[:10])

    ########### SELECTION: JETS ############
    jetPt  = arr["jetPt"]
    jetEta = arr["jetEta"]

    sel_jet = (jetPt >= jetPt_cut) & (abs(jetEta) < 2.4)

    ########### SELECTION: EVENTS ############
    total = ak.sum(weight)

    sel_ev[k] = arr["METNoMuTrigger"]
    sel_ev[k] = sel_ev[k] & (arr["met"] >= 200)
    sel_ev[k] = sel_ev[k] & (arr["Flag2_all"] == 1)

    accep_met[k] = ak.sum(weight[sel_ev[k]])

    print("trigger + MET",
          " total ", total,
          " passed ",accep_met[k],
          " frac ", accep_met[k]/total,
          ak.sum(sel_ev[k]))

    table["met"] += " & {0:6.3f} & {1:6.3f} & {2:6.2f}".format(
        100*accep_met[k]/total,
        100*accep_met[k]/total,
        accep_met[k]
    )

    acc_met = ak.sum(weight[sel_ev[k]])
    sel_temp = ak.sum(weight[sel_ev[k]])

    print("MET filters",
          ak.sum(weight[sel_ev[k]])/acc_met,
          ak.sum(weight[sel_ev[k]])/sel_temp,
          ak.sum(sel_ev[k]),
          ak.sum(weight[sel_ev[k]]))
    

    table["met_filter"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k]])/sel_temp,
        100*ak.sum(weight[sel_ev[k]])/acc_met,
        ak.sum(weight[sel_ev[k]])
    )

    sel_temp = ak.sum(weight[sel_ev[k]])

    ########### 0 leptons ##################
    has_lep = arr["nLeptons"] == 0
    #print(has_lep)
    sel_ev[k] = sel_ev[k] & has_lep
    print("0 leptons",
          ak.sum(weight[sel_ev[k]])/acc_met,
          ak.sum(weight[sel_ev[k]])/sel_temp,
          ak.sum(sel_ev[k]))

    table["lep"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k]])/sel_temp,
        100*ak.sum(weight[sel_ev[k]])/acc_met,
        ak.sum(weight[sel_ev[k]])
    )

    sel_temp = ak.sum(weight[sel_ev[k]])
    
    ########### JET REQUIREMENT ############
    has_jet = ak.sum(sel_jet, axis=1) >= 1
    sel_ev[k] = sel_ev[k] & has_jet

    print("1 jet",
          ak.sum(weight[sel_ev[k]])/acc_met,
          ak.sum(weight[sel_ev[k]])/sel_temp,
          ak.sum(sel_ev[k]))

    table["jet"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k]])/sel_temp,
        100*ak.sum(weight[sel_ev[k]])/acc_met,
        ak.sum(weight[sel_ev[k]])
    )

    sel_temp = ak.sum(weight[sel_ev[k]])

    ########### EVENT CLEANING ############
    sel_ev[k] = sel_ev[k] & (abs(arr["jetMet_dPhiMin"]) > 0.6)

    print("dphi(jet, met)",
          ak.sum(weight[sel_ev[k]])/acc_met,
          ak.sum(weight[sel_ev[k]])/sel_temp,
          ak.sum(sel_ev[k]))

    table["dphi_jetmet"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k]])/sel_temp,
        100*ak.sum(weight[sel_ev[k]])/acc_met,
        ak.sum(weight[sel_ev[k]])
    )

    sel_temp = ak.sum(weight[sel_ev[k]])

    sel_ev[k] = sel_ev[k] & (arr["nDtStations25"] < 3)

    print("DT station < 3",
          ak.sum(weight[sel_ev[k]])/acc_met,
          ak.sum(weight[sel_ev[k]])/sel_temp,
          ak.sum(sel_ev[k]))

    table["dt_station"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k]])/sel_temp,
        100*ak.sum(weight[sel_ev[k]])/acc_met,
        ak.sum(weight[sel_ev[k]])
    )

    sel_temp = ak.sum(weight[sel_ev[k]])

    sel_ev[k] = sel_ev[k] & (arr["nDtWheels25"] < 3)

    print("DT wheel < 3",
          ak.sum(weight[sel_ev[k]])/acc_met,
          ak.sum(weight[sel_ev[k]])/sel_temp,
          ak.sum(sel_ev[k]))

    table["dt_wheel"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k]])/sel_temp,
        100*ak.sum(weight[sel_ev[k]])/acc_met,
        ak.sum(weight[sel_ev[k]])
    )

    sel_temp = ak.sum(weight[sel_ev[k]])

    ########### CLUSTERS ############

    sel_rechitcluster = arr["dtRechitCluster_match_gLLP"]

    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    print("cluster efficiency",
          ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
          ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp)

    table["1_cluster"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

    sel_temp = ak.sum(weight[sel_ev[k] & has_cluster])

    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitClusterJetVetoPt"] < 20)
    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    print("jet cut",
          ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
          ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp)

    table["jet_veto"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

    sel_temp = ak.sum(weight[sel_ev[k] & has_cluster])

    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitClusterMuonVetoPt"] < 10)
    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    print("muon cut",
          ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
          ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp)

    table["muon_veto"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

    sel_temp = ak.sum(weight[sel_ev[k] & has_cluster])

    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitCluster_match_MB1hits_0p5"] <= 1)
    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    table["mb1"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitCluster_match_RPChits_dPhi0p5"] > 0)
    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    table["rpc"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitCluster_match_MB1hits_cosmics_minus"] <= 8)
    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitCluster_match_MB1hits_cosmics_plus"] <= 8)

    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    table["mb1_adjacent"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

    sel_rechitcluster = sel_rechitcluster & (abs(arr["dtRechitClusterMetEENoise_dPhi"]) < 1)
    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    table["dphi"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitClusterSize"] >= 100)
    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    table["nrechits"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

    sel_rechitcluster = sel_rechitcluster & (arr["dtRechitClusterMaxStation"] == 2)
    has_cluster = ak.sum(sel_rechitcluster, axis=1) >= 1

    table["mb2"] += " & {0:6.2f} & {1:6.2f} & {2:6.2f}".format(
        100*ak.sum(weight[sel_ev[k] & has_cluster])/sel_temp,
        100*ak.sum(weight[sel_ev[k] & has_cluster])/acc_met,
        ak.sum(weight[sel_ev[k] & has_cluster])
    )

for k, v in table.items():
    print(v + r" \\")